In [1]:
!pip install pandas



[notice] A new release of pip available: 22.2.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import sqlite3

In [8]:
df = pd.read_csv("data/sales.csv")
df

,Order_ID,Customer,Product,Sales,Order_Date
0,1001,John,Laptop,50000.0,01-05-2025
1,1002,Alex,Mouse,1000.0,02-05-2025
2,1003,Ram,NaN,2500.0,03-05-2025
3,1004,John,Laptop,50000.0,01-05-2025
4,1004,John,Laptop,50000.0,01-05-2025
5,1005,David,Keyboard,NaN,04-05-2025


In [27]:
# Check number of rows before cleaning

print("Rows before:", len(df))

# Remove duplicate rows

df = df.drop_duplicates()

# Check rows after cleaning

print("Rows after:", len(df))

# Display cleaned data

df

Rows before: 5
Rows after: 5


,Order_ID,Customer,Product,Sales,Order_Date
0,1001,John,Laptop,50000.0,2025-01-05
1,1002,Alex,Mouse,1000.0,2025-02-05
2,1003,Ram,Unknown,2500.0,2025-03-05
3,1004,John,Laptop,50000.0,2025-01-05
5,1005,David,Keyboard,0.0,2025-04-05


In [28]:
df.to_csv("output/cleaned_sales.csv", index=False)

print("Cleaned file created successfully")

Cleaned file created successfully


In [9]:
df.head()

,Order_ID,Customer,Product,Sales,Order_Date
0,1001,John,Laptop,50000.0,01-05-2025
1,1002,Alex,Mouse,1000.0,02-05-2025
2,1003,Ram,NaN,2500.0,03-05-2025
3,1004,John,Laptop,50000.0,01-05-2025
4,1004,John,Laptop,50000.0,01-05-2025


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Order_ID    6 non-null      int64  
 1   Customer    6 non-null      object 
 2   Product     5 non-null      object 
 3   Sales       5 non-null      float64
 4   Order_Date  6 non-null      object 
dtypes: float64(1), int64(1), object(3)
memory usage: 368.0+ bytes


In [11]:
df.isnull().sum()

Order_ID      0
Customer      0
Product       1
Sales         1
Order_Date    0
dtype: int64

In [13]:
df.drop_duplicates(inplace=True)

df

,Order_ID,Customer,Product,Sales,Order_Date
0,1001,John,Laptop,50000.0,01-05-2025
1,1002,Alex,Mouse,1000.0,02-05-2025
2,1003,Ram,NaN,2500.0,03-05-2025
3,1004,John,Laptop,50000.0,01-05-2025
5,1005,David,Keyboard,NaN,04-05-2025


In [14]:
df["Product"] = df["Product"].fillna("Unknown")

df

,Order_ID,Customer,Product,Sales,Order_Date
0,1001,John,Laptop,50000.0,01-05-2025
1,1002,Alex,Mouse,1000.0,02-05-2025
2,1003,Ram,Unknown,2500.0,03-05-2025
3,1004,John,Laptop,50000.0,01-05-2025
5,1005,David,Keyboard,NaN,04-05-2025


In [15]:
df["Sales"] = df["Sales"].fillna(0)

df

,Order_ID,Customer,Product,Sales,Order_Date
0,1001,John,Laptop,50000.0,01-05-2025
1,1002,Alex,Mouse,1000.0,02-05-2025
2,1003,Ram,Unknown,2500.0,03-05-2025
3,1004,John,Laptop,50000.0,01-05-2025
5,1005,David,Keyboard,0.0,04-05-2025


In [16]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"])
df

,Order_ID,Customer,Product,Sales,Order_Date
0,1001,John,Laptop,50000.0,2025-01-05
1,1002,Alex,Mouse,1000.0,2025-02-05
2,1003,Ram,Unknown,2500.0,2025-03-05
3,1004,John,Laptop,50000.0,2025-01-05
5,1005,David,Keyboard,0.0,2025-04-05


In [32]:
# Connect to SQLite database
connection = sqlite3.connect("database/sales.db")

# Replace old table with cleaned data
df.to_sql(
    "sales",
    connection,
    if_exists="replace",
    index=False
)

print("Cleaned data loaded successfully")

Cleaned data loaded successfully


In [17]:
df.to_csv("output/cleaned_sales.csv",index=False)

print("Cleaned file saved")

Cleaned file saved


In [29]:
connection = sqlite3.connect("database/sales.db")

df.to_sql("sales",connection,if_exists="replace",index=False)

print("Database updated successfully")

Database updated successfully


In [21]:
connection = sqlite3.connect("database/sales.db")
query = """SELECT SUM(Sales) AS TotalSales FROM sales"""

result = pd.read_sql(query,connection)

result

,TotalSales
0,103500.0


In [23]:
connection = sqlite3.connect("database/sales.db")
query = """SELECT AVG(Sales) AS TotalAVGSales FROM sales"""

result = pd.read_sql(query,connection)

result

,TotalAVGSales
0,20700.0


In [36]:
# Connect to database
connection = sqlite3.connect("database/sales.db")

# Remove old table
connection.execute("DROP TABLE IF EXISTS sales")

# Load cleaned dataframe again
df.to_sql("sales", connection, if_exists="replace", index=False)

print("Cleaned data loaded into database")

Cleaned data loaded into database


In [37]:
query = "SELECT * FROM sales"

new_data = pd.read_sql(query, connection)

new_data

,Order_ID,Customer,Product,Sales,Order_Date
0,1001,John,Laptop,50000.0,2025-01-05 00:00:00
1,1002,Alex,Mouse,1000.0,2025-02-05 00:00:00
2,1003,Ram,Unknown,2500.0,2025-03-05 00:00:00
3,1004,John,Laptop,50000.0,2025-01-05 00:00:00
4,1005,David,Keyboard,0.0,2025-04-05 00:00:00


In [38]:
query = """
SELECT
COUNT(Order_ID) AS TotalOrders,
SUM(Sales) AS TotalSales,
AVG(Sales) AS AverageSales,
MAX(Sales) AS HighestSale,
MIN(Sales) AS LowestSale
FROM sales
"""

kpi = pd.read_sql(query, connection)

kpi

,TotalOrders,TotalSales,AverageSales,HighestSale,LowestSale
0,5,103500.0,20700.0,50000.0,0.0


In [39]:
kpi.to_csv("output/kpi_data.csv", index=False)

print("KPI file updated successfully")

KPI file updated successfully


In [40]:
df.to_csv("output/cleaned_sales.csv", index=False)

print("Cleaned sales file saved")

Cleaned sales file saved
